# Lending Club Loans: Data Cleaning

Cleaning the Lending Club accepted-loans file (2007 to 2018, 2.26M rows) into a smaller table for MySQL and Power BI.

Output: `loans_clean.csv`, finished loans only, 36 columns.

In [ ]:
import glob
import os

import numpy as np
import pandas as pd

## Load data

In [ ]:
!pip install -q kagglehub
import kagglehub

path = kagglehub.dataset_download("wordsforthewise/lending-club")
RAW_FILE = [f for f in glob.glob(path + "/**/accepted*.csv.gz", recursive=True) if os.path.isfile(f)][0]
OUT_FILE = "loans_clean.csv"
REPORT_FILE = "cleaning_report.txt"
print(RAW_FILE)

Using Colab cache for faster access to the 'lending-club' dataset.
/kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz


## Columns and loan status

Using 29 of the 151 columns (loan terms, borrower profile, credit history, outcome). Loading everything needs too much RAM in Colab.

Only Fully Paid and Charged Off loans have a final outcome, so Current and Late loans are dropped. The two "Does not meet the credit policy" statuses are the same outcomes under an older label, so they get merged in.

In [ ]:
COLUMNS = [
    "id", "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "issue_d", "loan_status", "purpose", "addr_state",
    "dti", "delinq_2yrs", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_util", "total_pymnt", "total_rec_prncp",
    "total_rec_int", "recoveries", "application_type",
]

FINISHED_STATUS = {
    "Fully Paid": "Fully Paid",
    "Charged Off": "Charged Off",
    "Does not meet the credit policy. Status:Fully Paid": "Fully Paid",
    "Does not meet the credit policy. Status:Charged Off": "Charged Off",
}

report = []
def log(msg):
    print(msg)
    report.append(msg)

## Read in chunks, keep finished loans

In [ ]:
raw_rows = 0
status_counts = pd.Series(dtype="int64")
parts = []
for chunk in pd.read_csv(RAW_FILE, usecols=COLUMNS, dtype={"id": str},
                         chunksize=200_000, low_memory=False):
    raw_rows += len(chunk)
    status_counts = status_counts.add(chunk["loan_status"].value_counts(), fill_value=0)
    parts.append(chunk[chunk["loan_status"].isin(FINISHED_STATUS)])

df = pd.concat(parts, ignore_index=True)
del parts
df["loan_status"] = df["loan_status"].map(FINISHED_STATUS)

log(f"Raw rows read: {raw_rows:,}")
log("Raw loan_status counts:\n" + status_counts.astype(int).sort_values(ascending=False).to_string())
log(f"Rows after keeping finished loans only: {len(df):,}")

Raw rows read: 2,260,701
Raw loan_status counts:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Rows after keeping finished loans only: 1,348,059


## Drop empty rows and duplicates

In [ ]:
before = len(df)
df = df.dropna(subset=["loan_amnt", "issue_d"]).drop_duplicates(subset="id")
log(f"Removed {before - len(df):,} rows with no amount/date or duplicate id")

Removed 0 rows with no amount/date or duplicate id


## Fix types

- `term`: " 36 months" → 36
- `int_rate`, `revol_util`: "13.56%" → 13.56
- `issue_d`: "Dec-2015" → 2015-12-01
- `emp_length`: "10+ years" → 10, "< 1 year" → 0

In [ ]:
def to_number(s):
    return pd.to_numeric(s.astype(str).str.replace("%", "", regex=False).str.strip(), errors="coerce")

df["term_months"] = df["term"].str.extract(r"(\d+)", expand=False).astype(int)
df["int_rate"] = to_number(df["int_rate"])
df["revol_util"] = to_number(df["revol_util"])

# dates are "Dec-2015", or "Dec-15" if the file was opened in Excel
df["issue_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
short_year = df["issue_date"].isna()
df.loc[short_year, "issue_date"] = pd.to_datetime(df.loc[short_year, "issue_d"],
                                                  format="%b-%y", errors="coerce")
log(f"Issue dates that could not be read: {df['issue_date'].isna().sum():,}")
df = df.dropna(subset=["issue_date"])
df["issue_year"] = df["issue_date"].dt.year
df["issue_month"] = df["issue_date"].dt.month

emp = df["emp_length"].astype("string")
df["emp_length_yrs"] = np.where(emp.str.contains("<", na=False), 0,
                                pd.to_numeric(emp.str.extract(r"(\d+)", expand=False), errors="coerce"))

Issue dates that could not be read: 0


## Bad values

- DTI outside 0-100 (raw data has -1 and 999) → blank
- Income of 0 → blank. High incomes look real, so I kept them.
- Home ownership ANY / NONE → OTHER (too few rows)

In [ ]:
bad_dti = (df["dti"] < 0) | (df["dti"] > 100)
log(f"DTI values outside 0-100 set to blank: {bad_dti.sum():,}")
df.loc[bad_dti, "dti"] = np.nan

zero_inc = df["annual_inc"] <= 0
log(f"Annual income <= 0 set to blank: {zero_inc.sum():,}")
df.loc[zero_inc, "annual_inc"] = np.nan
log(f"Annual income 99.9th percentile: {df['annual_inc'].quantile(0.999):,.0f}")

df["home_ownership"] = df["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})

missing = df[["emp_length_yrs", "dti", "revol_util", "annual_inc", "inq_last_6mths"]].isna().sum()
log("Missing values left in key columns:\n" + missing.to_string())

DTI values outside 0-100 set to blank: 535
Annual income <= 0 set to blank: 361
Annual income 99.9th percentile: 580,000
Missing values left in key columns:
emp_length_yrs    78545
dti                 909
revol_util          897
annual_inc          365
inq_last_6mths       30


## New columns

- `is_default`: 1 if Charged Off
- `net_profit`: total paid minus amount funded
- `fico_avg` and income / DTI / FICO bands
- `is_matured`: 1 if the full term ended before the data cut-off (Mar 2019)

`is_matured` matters for comparing years. For recent loans, most of the finished ones are defaults, because the good loans are still running and got dropped. So 2016-2018 look riskier than they really are. Default rates should be compared on matured loans only.

In [ ]:
df["is_default"] = (df["loan_status"] == "Charged Off").astype(int)
df["net_profit"] = (df["total_pymnt"] - df["funded_amnt"]).round(2)
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

df["income_band"] = pd.cut(df["annual_inc"],
    bins=[0, 40_000, 60_000, 80_000, 100_000, 150_000, np.inf],
    labels=["<40K", "40-60K", "60-80K", "80-100K", "100-150K", "150K+"])
df["dti_band"] = pd.cut(df["dti"], bins=[-0.01, 10, 20, 30, 100],
    labels=["0-10", "10-20", "20-30", "30+"])
df["fico_band"] = pd.cut(df["fico_avg"], bins=[0, 660, 700, 740, 780, 900],
    labels=["<660", "660-699", "700-739", "740-779", "780+"])

CUTOFF = pd.Timestamp("2019-03-31")
maturity = df["issue_date"] + pd.to_timedelta(df["term_months"] * 30.44, unit="D")
df["is_matured"] = (maturity <= CUTOFF).astype(int)

## Save

In [ ]:
FINAL = [
    "id", "issue_date", "issue_year", "issue_month", "loan_amnt", "funded_amnt",
    "term_months", "int_rate", "installment", "grade", "sub_grade", "purpose",
    "loan_status", "is_default", "is_matured", "total_pymnt", "total_rec_prncp",
    "total_rec_int", "recoveries", "net_profit", "annual_inc", "income_band",
    "emp_length_yrs", "home_ownership", "verification_status", "addr_state",
    "dti", "dti_band", "fico_avg", "fico_band", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_util", "application_type",
]
clean = df[FINAL].copy()
clean["issue_date"] = clean["issue_date"].dt.strftime("%Y-%m-%d")
clean.to_csv(OUT_FILE, index=False)
print(f"Saved {OUT_FILE}: {len(clean):,} rows, {clean.shape[1]} columns")

Saved loans_clean.csv: 1,348,059 rows, 36 columns


## Summary check

In [ ]:
log("\n===== CLEAN DATA SUMMARY =====")
log(f"Rows: {len(clean):,}   Columns: {clean.shape[1]}")
log(f"Issue dates: {clean['issue_date'].min()} to {clean['issue_date'].max()}")
log(f"Total funded: ${clean['funded_amnt'].sum():,.0f}")
log(f"Overall default rate: {clean['is_default'].mean():.2%}")
log(f"Default rate, matured loans only: {clean.loc[clean.is_matured == 1, 'is_default'].mean():.2%}")
log("Default rate by grade:\n" + clean.groupby("grade")["is_default"].mean().map("{:.1%}".format).to_string())
log("Loans by issue year:\n" + clean["issue_year"].value_counts().sort_index().to_string())

with open(REPORT_FILE, "w") as f:
    f.write("\n".join(report))


===== CLEAN DATA SUMMARY =====
Rows: 1,348,059   Columns: 36
Issue dates: 2007-06-01 to 2018-12-01
Total funded: $19,412,303,675
Overall default rate: 19.98%
Default rate, matured loans only: 15.19%
Default rate by grade:
grade
A     6.0%
B    13.4%
C    22.4%
D    30.4%
E    38.4%
F    45.1%
G    49.7%
Loans by issue year:
issue_year
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134804
2014    223102
2015    375545
2016    293095
2017    169300
2018     56311


## Copy to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/bank-loan
!cp loans_clean.csv cleaning_report.txt /content/drive/MyDrive/bank-loan/
print("Copied to My Drive/bank-loan")

Mounted at /content/drive
Copied to My Drive/bank-loan
